# Schema Intelligence: Naive Agent Version

Agentic relational-database understanding + SQL generation over a synthetic e-commerce PostgreSQL 17 database.


## Environment setup and dependencies

In [1]:
import json, os, sys
from pathlib import Path as _Path
_here = _Path.cwd()
_root = _here if (_here / 'src').exists() else _here.parent
sys.path.insert(0, str(_root / 'src'))
os.chdir(str(_root))

from dotenv import load_dotenv
load_dotenv()

from urllib.parse import urlparse
import importlib.metadata as md

for pkg in ['psycopg', 'python-dotenv', 'numpy', 'sentence-transformers', 'openai']:
    try:
        print(pkg, md.version(pkg))
    except Exception:
        print(pkg, 'MISSING')

print('python', sys.version.split()[0])
print('POSTGRES:', os.getenv('POSTGRES_HOST'), os.getenv('POSTGRES_PORT'), os.getenv('POSTGRES_DB'), os.getenv('POSTGRES_USER'))
print('ORION_BASE:', urlparse(os.getenv('ORION_BASE_URL', '')).hostname)
print('ORION_MODEL:', os.getenv('ORION_MODEL'))
print('ORION_API_KEY present:', bool(os.getenv('ORION_API_KEY')))

psycopg 3.3.9
python-dotenv 1.2.4
numpy 2.5.5
sentence-transformers 6.3.0
openai 3.51.0
python 3.13.7
POSTGRES: localhost 5433 schema_intelligence schema_admin
ORION_BASE: api.orion.cmkl.ai
ORION_MODEL: Mango 1.5-9B
ORION_API_KEY present: True


## Environment variables and Orion API configuration

In [2]:
from agent import get_client

client = get_client()
print('models:', [m.id for m in client.models.list().data])

/Users/dannykhant/Code/cmkl/assignments/schema-intelligence-agent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1124 00:34:34.135000 34613 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


models: ['Qwen 3.8-Flash-Next', 'Qwen 3.8-27B', 'Mango 1.5-9B']


## PostgreSQL connection test

In [3]:
import os
import psycopg

conn = psycopg.connect(
    host=os.getenv('POSTGRES_HOST'), 
    port=os.getenv('POSTGRES_PORT'), 
    dbname=os.getenv('POSTGRES_DB'), 
    user=os.getenv('POSTGRES_USER'), 
    password=os.getenv('POSTGRES_PASSWORD')
)
tables = [r[0] for r in conn.execute("SELECT tablename FROM pg_tables WHERE schemaname='public' ORDER BY tablename").fetchall()]
print('tables:', tables)

for t in tables:
    print(t, conn.execute('SELECT count(*) FROM ' + t).fetchone()[0])
n_fk = conn.execute("SELECT count(*) FROM information_schema.table_constraints WHERE constraint_type='FOREIGN KEY' AND table_schema='public'").fetchone()[0]

print('foreign keys:', n_fk)
conn.close()

tables: ['categories', 'customers', 'order_items', 'orders', 'payments', 'products']
categories 4
customers 8
order_items 17
orders 12
payments 12
products 10
foreign keys: 5


## Tools: Database schema inspection (`get_table_schema`)

In [4]:
from db_tools import get_table_schema

s = get_table_schema('customers')

for c in s['columns']:
    print(c['column_name'], '|', c['data_type'], '| nullable:', c['is_nullable'])

print('constraints:', [(k['constraint_type'], k['column_name']) for k in s['constraints']])

for t in ['orders', 'order_items', 'products', 'categories', 'payments']:
    r = get_table_schema(t)
    print(t + ':', str(len(r['columns'])) + ' cols, ' + str(len(r['constraints'])) + ' constraints')

customer_id | bigint | nullable: NO
full_name | character varying | nullable: NO
email | character varying | nullable: NO
created_at | timestamp without time zone | nullable: NO
is_active | boolean | nullable: NO
constraints: [('PRIMARY KEY', 'customer_id'), ('UNIQUE', 'email')]
orders: 4 cols, 2 constraints


order_items: 5 cols, 3 constraints


products: 5 cols, 2 constraints


categories: 3 cols, 2 constraints


payments: 5 cols, 2 constraints


## Tools: Relationship discovery (`find_relationships`, BFS over the FK graph)

In [5]:
from db_tools import find_relationships

for a, b in [('customers', 'products'), ('orders', 'payments'), ('categories', 'customers')]:
    r = find_relationships(a, b)
    print(a, '->', b, ':', r['path'])
    for j in r.get('joins', []):
        print('   ', j['left_table'] + '.' + j['left_column'], '=', j['right_table'] + '.' + j['right_column'])

customers -> products : ['customers', 'orders', 'order_items', 'products']
    customers.customer_id = orders.customer_id
    orders.order_id = order_items.order_id
    order_items.product_id = products.product_id


orders -> payments : ['orders', 'payments']
    orders.order_id = payments.order_id


categories -> customers : ['categories', 'products', 'order_items', 'orders', 'customers']
    categories.category_id = products.category_id
    products.product_id = order_items.product_id
    order_items.order_id = orders.order_id
    orders.customer_id = customers.customer_id


## RAG document loading and embedding generation

18 Markdown docs, one chunk each (short docs), MiniLM-L6-v2 normalized embeddings, in-memory NumPy index. This cell downloads/loads the model on first run (~1 min).

In [6]:
import numpy as np
from rag import SchemaKnowledgeBase

kb = SchemaKnowledgeBase()
print('chunks:', len(kb))
print('embeddings:', kb.embeddings.shape)

n = np.linalg.norm(kb.embeddings, axis=1)
print('norm min/max:', round(float(n.min()), 4), round(float(n.max()), 4))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11550.85it/s]

chunks: 18
embeddings: (18, 384)
norm min/max: 1.0 1.0


## Tools: Semantic document retrieval (`search_schema_docs`, top-3 cosine)

In [7]:
questions = [
    'What is the difference between a completed order and a paid order?', 
    'Should I use current product price or historical purchase price?', 
    'How to avoid join multiplication when aggregating payments and order items?'
]

for q in questions:
    print('Q:', q)
    for r in kb.search_schema_docs(q):
        print('   [%.3f]' % r['similarity'], r['source'], '(' + r['category'] + ')')

Q: What is the difference between a completed order and a paid order?
   [0.556] business_definitions/order_status.md (business_definitions)
   [0.543] business_rules/order_eligibility.md (business_rules)
   [0.516] business_rules/sales_calculation.md (business_rules)
Q: Should I use current product price or historical purchase price?
   [0.549] data_semantics/product_pricing.md (data_semantics)
   [0.537] query_guidance/purchase_history.md (query_guidance)
   [0.306] query_guidance/category_sales.md (query_guidance)
Q: How to avoid join multiplication when aggregating payments and order items?
   [0.636] business_rules/sales_calculation.md (business_rules)
   [0.496] business_definitions/customer_spending.md (business_definitions)
   [0.425] query_guidance/category_sales.md (query_guidance)


## 9. Tool definitions and JSON schemas

In [8]:
import json
from tools import build_tool_definitions, execute_tool

defs = build_tool_definitions()

print('tools:', [d['function']['name'] for d in defs])
print(json.dumps(defs[0], indent=2))
print('direct dispatch demo:', execute_tool('get_table_schema', {'table_name': 'orders'}, kb)[:200])

tools: ['get_table_schema', 'find_relationships', 'search_schema_docs', 'validate_sql']
{
  "type": "function",
  "function": {
    "name": "get_table_schema",
    "description": "Get column names, data types, primary keys, foreign keys, and unique constraints for one or more allowed tables. Prefer a single call listing every table you need over one call per table.",
    "parameters": {
      "type": "object",
      "properties": {
        "table_names": {
          "type": "array",
          "items": {
            "type": "string",
            "enum": [
              "customers",
              "orders",
              "order_items",
              "products",
              "categories",
              "payments"
            ]
          },
          "minItems": 1,
          "description": "Tables to inspect, e.g. [\"orders\", \"payments\"]."
        }
      },
      "required": [
        "table_names"
      ],
      "additionalProperties": false
    }
  }
}
direct dispatch demo: {"table":

## 10. Tool-calling agent implementation

Loop: system + user message, `tool_choice: auto`, execute requested tools, append `role: tool` results, repeat. Max 5 iterations; no ReAct, no retries, no memory, no SQL execution. Source shown directly from `src/agent.py` so the notebook never drifts from the module.

In [9]:
import inspect
from agent import SYSTEM_PROMPT, run_agent

print(SYSTEM_PROMPT)
print('\n===== run_agent source =====\n')
print(inspect.getsource(run_agent))

You are a Schema Intelligence Agent that helps users
understand a PostgreSQL database and generate SQL queries.

You can inspect table schemas, discover relationships,
retrieve business documentation, and validate SQL.

Use database metadata to determine actual table structures
and foreign-key relationships. When you need several tables,
fetch them all in one get_table_schema call.

Use the knowledge base to understand business definitions,
data semantics, and business rules.

When generating SQL:
1. Use only tables and columns confirmed by schema inspection.
2. Follow the retrieved business definitions.
3. Discover relationships when necessary.
4. Generate SQL that answers the user's question.
5. Validate the SQL using the validate_sql tool.

Never execute generated SQL.
Never invent database columns or relationships.
Explain important assumptions when the question is ambiguous.

If validation fails, report the failure rather than
automatically attempting to repair the SQL.

===== run

## 11. Evaluation queries (live: Mango 1.5-9B with tools)

8 questions in 3 sets, written as backend-developer tasks rather than analytics: 

A — schema and constraints (what the database accepts) · B — writing production queries · C — debugging and performance. 

Tool calls are recorded live; `independent check` re-validates any ```sql fences (notebook-side, not a model call).

In [10]:
import json, re
import agent as agent_mod
from agent import run_agent
from db_tools import validate_sql

TRANSCRIPTS, QUERY_LOG = [], []
QUESTION_IDS = ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2']
TRANSCRIPTS_PATH = _root / 'evaluation' / 'transcripts.json'
_real_exec = agent_mod.execute_tool

def _rec(name, args, kb_):
    out = _real_exec(name, args, kb_)

    try:
        a = json.loads(args) if isinstance(args, str) else dict(args)
    except Exception:
        a = {'_raw': str(args)[:200]}

    TRANSCRIPTS.append({'name': name, 'args': a, 'preview': out[:400]})

    return out


agent_mod.execute_tool = _rec

def independent_sql_check(answer):
    # A model turn with no tool calls and no content returns None; record it
    # as a finding instead of crashing the eval loop.
    if not isinstance(answer, str) or not answer.strip():
        print('[independent check] no answer text (' + repr(answer) + ')')
        return []

    fences = re.findall(r'```sql(.*?)```', answer, re.S | re.I)
    results = []

    for i, f in enumerate(fences):
        v = validate_sql(f)
        results.append({'valid': v['valid'], 'message': v['message'][:200]})
        print('[independent check] sql[' + str(i) + '] valid=' + str(v['valid']) + ' :: ' + v['message'][:200])
        
    if not fences:
        print('[independent check] no ```sql fences in answer')

    return results
    

def run_eval(q):
    start = len(TRANSCRIPTS)
    ans = run_agent(q, kb, client=client)
    calls = TRANSCRIPTS[start:]

    print('QUERY:', q)
    print('tool calls:', len(calls))

    for c in calls:
        print('-', c['name'], json.dumps(c['args'])[:200])

    print('---- ANSWER ----')
    if ans is None:
        print('[note] model returned None (no content, no tool calls)')
    else:
        print(ans)

    checks = independent_sql_check(ans)
    QUERY_LOG.append({'id': QUESTION_IDS[len(QUERY_LOG)], 'query': q,
                      'tools': [c['name'] for c in calls], 'answer': ans,
                      'none_answer': ans is None, 'sql_checks': checks})

    # Persist after every question so the evaluator can run standalone.
    TRANSCRIPTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    TRANSCRIPTS_PATH.write_text(json.dumps(QUERY_LOG, indent=2), encoding='utf-8')

    return ans

In [11]:
import json

GT_PATH = _root / 'evaluation' / 'ground_truth.json'
GROUND_TRUTH = json.loads(GT_PATH.read_text())
print('ground truth:', GROUND_TRUTH['generated_at'], '|', GT_PATH.relative_to(_root))

def summarise(name, e):
    if name == 'A1':
        return f"amount NOT NULL; status in {e['allowed_payment_status']}; {len(e['columns'])} columns"
    if name == 'A2':
        return '; '.join(f"{r['table']}.{r['column']} ({r['on_delete']})" for r in e['referencing_tables'])
    if name == 'A3':
        return f"nullable={e['nullable']}; SUM needs NULL handling={e['sum_needs_null_handling']}"
    if name == 'B1':
        return f"{e['row_count']} rows, one per order (order 1 = {e['rows'][0]['total_item_value']})"
    if name == 'B2':
        return f"{e['row_count']} rows of {e['total_customers_in_table']} customers"
    if name == 'B3':
        return f"after {e['cursor']} -> order_ids {[r['order_id'] for r in e['rows']]}"
    if name == 'C1':
        return f"buggy {e['buggy_row_count']} rows -> fixed {e['fixed_row_count']} rows"
    if name == 'C2':
        return f"uses_index={e['uses_index']} ({e['indexes'][0]['index']}, {e['indexes'][0]['scan_direction']})"
    return str(e)

for name, q in GROUND_TRUTH['questions'].items():
    print(f"{name} | {summarise(name, q['expected'])}")


ground truth: 2026-11-23T09:54:40+00:00 | evaluation/ground_truth.json
A1 | amount NOT NULL; status in ['failed', 'paid', 'pending', 'refunded']; 5 columns
A2 | order_items.order_id (NO ACTION); payments.order_id (NO ACTION)
A3 | nullable=False; SUM needs NULL handling=False
B1 | 12 rows, one per order (order 1 = 154.99)
B2 | 0 rows of 8 customers
B3 | after {'order_date': '2026-10-07', 'order_id': 3} -> order_ids [11, 6, 9]
C1 | buggy 3 rows -> fixed 2 rows
C2 | uses_index=True (idx_orders_order_date, Backward)


**Ground truth must be regenerated after re-seeding.**

`evaluation/ground_truth.json` is generated from the live database by `evaluation/build_ground_truth.py`; it is not written by hand. The seed data uses `CURRENT_TIMESTAMP`, so recreating the volume (`docker compose down -v && docker compose up -d`) shifts every order date and makes the stored expectations stale — including B3's fixed cursor date. Run this before trusting the grades:

```bash
.venv/bin/python evaluation/build_ground_truth.py
```


### 11a: Schema and constraints (onboarding a new service)


Backend questions about what the database will actually accept. Note what each tool can and cannot see: `get_table_schema` returns columns, nullability, PK/FK/UNIQUE but **not** CHECK constraints or indexes — so allowed `payment_status` values must come from the knowledge base, and index usage must come from `EXPLAIN`.

In [12]:
schema_questions = [
    # Needs: get_table_schema (columns/nullability) + search_schema_docs (allowed status values live in a CHECK constraint, not exposed by the tool)
    "What columns, types and constraints does the payments table have? I need to know whether amount is nullable and which payment_status values are allowed.",
    # Needs: get_table_schema (FKs on orders) — trap: inventing an ON DELETE CASCADE that does not exist in the schema
    "I'm writing a delete-order endpoint. Which tables and columns reference orders, and what should I handle before deleting an order row?",
    # Needs: get_table_schema (is_nullable) — trap: answering 'yes, use COALESCE' when the column is NOT NULL
    "Is order_items.unit_price_at_purchase nullable? I want to SUM it in a revenue report — will NULLs be a problem?",
]
for q in schema_questions:
    _ = run_eval(q)
    print('\n' + '=' * 60 + '\n')

QUERY: What columns, types and constraints does the payments table have? I need to know whether amount is nullable and which payment_status values are allowed.
tool calls: 1
- get_table_schema {"table_names": ["payments"]}
---- ANSWER ----


The **payments** table has the following schema:

| Column | Data Type | Nullable | Constraints |
|--------|-----------|----------|-------------|
| `payment_id` | bigint | NO | PRIMARY KEY |
| `order_id` | bigint | NO | FOREIGN KEY → `orders(order_id)` |
| `payment_date` | timestamp without time zone | NO | |
| `amount` | numeric | NO | |
| `payment_status` | character varying | NO | |

**Summary of your questions:**

- **Is `amount` nullable?** No, `amount` is defined as `NOT NULL`.
- **What `payment_status` values are allowed?** The schema does not specify any CHECK constraints or domain restrictions on `payment_status`. It's a text column (`character varying`) with no explicit constraints shown in the table definition.

The database doesn't show

QUERY: I'm writing a delete-order endpoint. Which tables and columns reference orders, and what should I handle before deleting an order row?
tool calls: 5
- get_table_schema {"table_names": ["orders", "order_items", "payments", "customers", "products", "categories"]}
- find_relationships {"source_table": "orders", "target_table": "order_items"}
- find_relationships {"source_table": "orders", "target_table": "payments"}
- search_schema_docs {"query": "order delete cascade foreign key constraints business rules"}
- search_schema_docs {"query": "cancel order delete transaction soft delete audit trail"}
---- ANSWER ----


## Tables & Columns Referencing Orders

**1. `order_items`**
| Column | FK Reference |
|--------|-------------|
| `order_id` | `orders.order_id` |

**2. `payments`**
| Column | FK Reference |
|--------|-------------|
| `order_id` | `orders.order_id` |

---

## What to Handle Before Deleting an Order

### 1. **Foreign Key Deletion Handling**

You have two options for the 

QUERY: Is order_items.unit_price_at_purchase nullable? I want to SUM it in a revenue report — will NULLs be a problem?
tool calls: 2
- get_table_schema {"table_names": ["order_items"]}
- search_schema_docs {"query": "order_items unit_price_at_purchase not null constraints business rules"}
---- ANSWER ----


**No, `order_items.unit_price_at_purchase` is NOT NULL.** It has `"is_nullable": "NO"` at the database level, so NULLs will **not** be a problem for your SUM query.

**Key points:**
- The column is defined with `NOT NULL` constraint, so every row has a value
- No need for `COALESCE` or `IFNULL`
- The data semantics confirm this is the historical price at time of purchase, which is the correct field for revenue calculations

**For your revenue report, the typical pattern would be:**

```sql
SELECT ORDER_ID,
       SUM(quantity * unit_price_at_purchase) AS total_revenue
FROM order_items
GROUP BY ORDER_ID;
```

No special handling for NULLs required.
[independent check] sql[0] valid=Tr

### 11b: Writing production queries


The core task: author SQL a backend developer could ship. Each has a semantic trap that a validating query can still get wrong.

In [13]:
authoring_questions = [
    # Needs: search_schema_docs (join multiplication) + get_table_schema + validate_sql — trap: joining payments and order_items directly multiplies order totals
    "Write a query for the order-history endpoint: each order with its total item value. An order can have multiple payments, and the totals must not be duplicated.",
    # Needs: get_table_schema + validate_sql — trap: LEFT JOIN ... IS NULL vs NOT IN with NULLs
    "Write a query returning customers who have never placed an order, with id, name and email.",
    # Needs: get_table_schema + validate_sql — trap: OFFSET-based paging, or no tiebreaker so pages can repeat/skip rows
    "Write a keyset-pagination query for the orders table: the next 5 orders after the cursor (order_date = 2026-10-07, order_id = 3), newest first, stable across pages.",
]
for q in authoring_questions:
    _ = run_eval(q)
    print('\n' + '=' * 60 + '\n')

QUERY: Write a query for the order-history endpoint: each order with its total item value. An order can have multiple payments, and the totals must not be duplicated.
tool calls: 3
- get_table_schema {"table_names": ["orders", "order_items", "products", "categories", "payments"]}
- search_schema_docs {"query": "order history total item value payment"}
- validate_sql {"sql": "WITH order_totals AS (\n    SELECT \n        order_id,\n        SUM(quantity * unit_price_at_purchase) AS total_item_value\n    FROM order_items\n    GROUP BY order_id\n)\nSELECT \n    o.orde
---- ANSWER ----


## Order History Query

```sql
WITH order_totals AS (
    SELECT 
        order_id,
        SUM(quantity * unit_price_at_purchase) AS total_item_value
    FROM order_items
    GROUP BY order_id
)
SELECT 
    o.order_id,
    o.order_date,
    o.status,
    ot.total_item_value,
    SUM(p.amount) AS total_payment_amount
FROM orders o
LEFT JOIN order_totals ot ON o.order_id = ot.order_id
LEFT JOIN payments p ON 

QUERY: Write a query returning customers who have never placed an order, with id, name and email.
tool calls: 2
- get_table_schema {"table_names": ["customers", "orders"]}
- validate_sql {"sql": "SELECT c.customer_id, c.full_name, c.email\nFROM customers c\nLEFT JOIN orders o ON c.customer_id = o.customer_id\nWHERE o.customer_id IS NULL"}
---- ANSWER ----


Here's the query to find customers who have never placed an order:

```sql
SELECT c.customer_id, c.full_name, c.email
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL
```

**How it works:**
- The `LEFT JOIN` finds all customers and matches them with their orders
- The `WHERE o.customer_id IS NULL` filters to only customers with no matching order in the orders table
- This returns `customer_id`, `full_name`, and `email` for those customers

The PostgreSQL plan shows an efficient **Hash Right Anti Join**, which efficiently identifies customers without orders with minimal cost.
[independe

QUERY: Write a keyset-pagination query for the orders table: the next 5 orders after the cursor (order_date = 2026-10-07, order_id = 3), newest first, stable across pages.
tool calls: 2
- get_table_schema {"table_names": ["orders"]}
- validate_sql {"sql": "SELECT *\nFROM orders\nWHERE order_date > '2026-10-07' \n   OR (order_date = '2026-10-07' AND order_id > 3)\nORDER BY order_date DESC, order_id DESC\nLIMIT 5;"}
---- ANSWER ----


## Keyset-Pagination Query for Orders

```sql
SELECT *
FROM orders
WHERE order_date > '2026-10-07' 
   OR (order_date = '2026-10-07' AND order_id > 3)
ORDER BY order_date DESC, order_id DESC
LIMIT 5;
```

### How it works:

| Condition | Logic |
|-----------|-------|
| `order_date > '2026-10-07'` | All orders after the cursor's date (newer dates) |
| `order_date = '2026-10-07' AND order_id > 3` | Orders on the same date but after the cursor's row |

### Ordering (newest first):
- **Primary**: `order_date DESC` — latest dates first
- **Tiebreaker**: `order_i

### 11c: Debugging and performance


Two questions a developer asks about code that already exists: why is this wrong, and will this be fast.

In [14]:
debug_questions = [
    # Needs: search_schema_docs (fan-out) + validate_sql — diagnosis must name row multiplication, fix must pre-aggregate
    "This query returns duplicate payment amounts per order:\nSELECT o.order_id, p.amount FROM orders o JOIN payments p ON p.order_id = o.order_id JOIN order_items oi ON oi.order_id = o.order_id WHERE o.customer_id = 1;\nWhy does it return duplicates, and how do I fix it?",
    # Needs: validate_sql (EXPLAIN plan) — the tool does not expose indexes, so the answer must come from reading the plan
    "Will this query use the index on order_date? Read the execution plan and tell me.\nSELECT order_id, order_date FROM orders WHERE order_date >= CURRENT_DATE - INTERVAL '30 days' ORDER BY order_date DESC LIMIT 20;",
]
for q in debug_questions:
    _ = run_eval(q)
    print('\n' + '=' * 60 + '\n')

QUERY: This query returns duplicate payment amounts per order:
SELECT o.order_id, p.amount FROM orders o JOIN payments p ON p.order_id = o.order_id JOIN order_items oi ON oi.order_id = o.order_id WHERE o.customer_id = 1;
Why does it return duplicates, and how do I fix it?
tool calls: 2
- get_table_schema {"table_names": ["orders", "payments", "order_items"]}
- get_table_schema {"table_names": ["customers"]}
---- ANSWER ----


The query returns duplicates because you're joining a **one-to-many relationship** (orders → payments) with **another one-to-many relationship** (orders → order_items). This creates a Cartesian product: if an order has 3 items and 2 payments, you get 3 × 2 = 6 rows.

The `order_items` join is not needed for your goal of getting the payment amount per order.

## Fix

Remove the unnecessary `order_items` join and use `DISTINCT` to deduplicate:

```sql
SELECT DISTINCT o.order_id, p.amount 
FROM orders o 
JOIN payments p ON p.order_id = o.order_id 
WHERE o.customer_id

QUERY: Will this query use the index on order_date? Read the execution plan and tell me.
SELECT order_id, order_date FROM orders WHERE order_date >= CURRENT_DATE - INTERVAL '30 days' ORDER BY order_date DESC LIMIT 20;
tool calls: 1
- validate_sql {"sql": "SELECT order_id, order_date FROM orders WHERE order_date >= CURRENT_DATE - INTERVAL '30 days' ORDER BY order_date DESC LIMIT 20;"}
---- ANSWER ----


Yes, the query **will use the index on order_date**.

Looking at the execution plan, here's what it shows:

**Key Plan Details:**

| Node | Details |
|------|---------|
| **Plan Type** | `Index Scan` |
| **Index Name** | `idx_orders_order_date` |
| **Scan Direction** | `Backward` |
| **Index Cond** | `(order_date >= (CURRENT_DATE - '30 days'::interval))` |

**What this means:**

1. **Index is being used**: The planner selected an `Index Scan` rather than a sequential or random scan
2. **Direction matches ORDER BY**: The "Backward" scan direction is optimal for your `ORDER BY order_date D

## Results and failure analysis

Grades are computed by `evaluation/evaluate.py`, not written by hand: the agent's own SQL is executed read-only and compared row-by-row to `evaluation/ground_truth.json`, prose answers go to an LLM judge, and cosine similarity plus keyword checks are reported as flags. The cells below print the verdicts and the computed failure analysis.

In [15]:
import sys, importlib
sys.path.insert(0, str(_root / 'evaluation'))
import evaluate as evaluator
importlib.reload(evaluator)

from IPython.display import Markdown, display

RESULTS = evaluator.evaluate(QUERY_LOG)

# One column per signal: executed-SQL evidence, cosine, verdict, summary.
display(Markdown(evaluator.to_markdown(RESULTS)))

import json as _json
(_root / 'evaluation' / 'results.json').write_text(_json.dumps({
    'transcripts': str(_root / 'evaluation' / 'transcripts.json'),
    'cosine_min': evaluator.COSINE_SIMILAR_MIN,
    'summary': evaluator.summarise(RESULTS),
    'results': RESULTS,
}, indent=2, default=str), encoding='utf-8')

counts = evaluator.summarise(RESULTS)
print('score:', ', '.join(f'{v} {k}' for k, v in sorted(counts.items())), f'(of {len(RESULTS)})')
print('grading: SQL result comparison where the answer contains SQL, keyword check otherwise; cosine is a flag.')

| Q | calls | evidence | cos | verdict | summary |
|---|---|---|---|---|---|
| A1 | 1 | keywords missing: paid | 0.79 | **incorrect** | keyword check · missing: paid; failed; refunded |
| A2 | 5 | keywords missing: no cascade / not cascade / no on delet | 0.84 | **incorrect** | keyword check · missing: no cascade / not cascade / no on delete cascade / without cascade |
| A3 | 2 | keywords pass | 0.80 | **correct** | keyword check |
| B1 | 3 | SQL rows match (12) | 0.85 | **correct** | sql result comparison |
| B2 | 2 | SQL rows match (0) | 0.79 | **correct** | sql result comparison |
| B3 | 2 | SQL mismatch (5 vs 3 rows) | 0.76 | **incorrect** | sql result comparison |
| C1 | 2 | SQL rows match (2) | 0.83 | **correct** | sql result comparison |
| C2 | 1 | keywords pass | 0.80 | **correct** | keyword check |

<IPython.core.display.Markdown object>
score: 5 correct, 3 incorrect (of 8)
grading: SQL result comparison where the answer contains SQL, keyword check otherwise; cosine is a flag.


In [16]:
# Failure analysis, computed from RESULTS — no hand-written grades.
sql_decided = [r for r in RESULTS if r['basis'] == 'sql result comparison']
kw_decided = [r for r in RESULTS if r['basis'] == 'keyword check']

print(f"{len(sql_decided)}/{len(RESULTS)} graded by executing the generated SQL and comparing rows "
      f"to ground truth; {len(kw_decided)} by keyword checks on prose answers.")
print()

for r in RESULTS:
    if r['verdict'] == 'incorrect':
        print(f"[INCORRECT] {r['question']} — {r['basis']}")
        if r['sql_check'] and r['sql_check'].get('attempts'):
            for a in r['sql_check']['attempts'][-1:]:
                if 'detail' in a:
                    print(f"   expected {a['detail']['expected']}")
                    print(f"   actual   {a['detail']['actual']}")
                elif 'error' in a:
                    print(f"   sql error: {a['error']}")
        for f in r['flags']:
            print(f"   {f}")
        print()

4/8 graded by executing the generated SQL and comparing rows to ground truth; 4 by keyword checks on prose answers.

[INCORRECT] A1 — keyword check
   missing: paid; failed; refunded

[INCORRECT] A2 — keyword check
   missing: no cascade / not cascade / no on delete cascade / without cascade

[INCORRECT] B3 — sql result comparison
   expected [(11, '2026-09-17'), (6, '2026-04-30'), (9, '2026-01-20')]
   actual   [(12, '2026-11-13'), (8, '2026-11-11'), (4, '2026-11-08'), (2, '2026-11-06'), (10, '2026-11-04')]

